# RQ3 — Meta-Learner Training & Evaluation

**Research Question**: Can a meta-learner predict the best clustering method from dataset properties alone, before any clustering is run?

## Primary Framework: Regression-First

The meta-learner predicts a **vector of 6 LSE values** for a new dataset.
The recommended method is the argmax of that predicted vector.
This is more informative than direct classification: it tells you not just
which method to pick, but also how much better it is expected to be.

## Two Regression Architectures

- **Architecture A (multi-output)**: one regressor with 6 outputs jointly
- **Architecture B (per-method, de Souto style)**: 6 separate single-output regressors

## Four Evaluation Metrics

| Metric | What it measures |
|--------|------------------|
| Per-method MAE | Regression quality (primary) |
| Spearman Rank Correlation (SRC) | Whole-ranking quality, de Souto 2008 |
| Top-1 accuracy | Did argmax match the true best? |
| Top-2 + tie zone (0.02) | True best in top-2 OR within 0.02 LSE of predicted top |

## Three Baselines

| Baseline | Description |
|----------|-------------|
| Always k-means | Current lower bound |
| **de Souto default-ranking** | Always predict the method with highest avg LSE across training (the real baseline to beat) |
| Oracle | Run all 6 methods, pick actual best (upper bound) |

**Outputs**: `outputs/models/meta_clf_optA.pkl`, `meta_reg_optA.pkl`, ablation figures

In [1]:
import os, sys, pickle, warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.metrics import confusion_matrix, classification_report

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, os.path.join(ROOT, 'src'))

from meta_learner import (
    extract_Xy_clf, extract_Xy_reg,
    loo_classify, loo_regress, loo_regress_per_method,
    baseline_always, default_ranking_baseline, oracle_expected_lse,
    report_clf_results, report_reg_results,
    build_classifier_candidates, build_regressor_candidates,
    LSE_COLS, METHOD_NAMES,
)

META_DIR   = os.path.join(ROOT, 'data', 'meta_table')
MODELS_DIR = os.path.join(ROOT, 'outputs', 'models')
FIGS_DIR   = os.path.join(ROOT, 'outputs', 'figures')
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(FIGS_DIR,   exist_ok=True)

SEED = 42
np.random.seed(SEED)
print('Imports OK')

Imports OK


In [2]:
df_a    = pd.read_csv(os.path.join(META_DIR, 'meta_training_optA.csv'))
df_b    = pd.read_csv(os.path.join(META_DIR, 'meta_training_optB.csv'))
df_c    = pd.read_csv(os.path.join(META_DIR, 'meta_training_optC.csv'))
df_ab   = pd.read_csv(os.path.join(META_DIR, 'meta_training_optAB.csv'))
df_ac   = pd.read_csv(os.path.join(META_DIR, 'meta_training_optAC.csv'))
df_rand = pd.read_csv(os.path.join(META_DIR, 'meta_training_optRand.csv'))

for name, df in [('A', df_a), ('B', df_b), ('C', df_c),
                  ('AB', df_ab), ('AC', df_ac), ('Rand', df_rand)]:
    df.dropna(subset=['best_method'], inplace=True)
    df.reset_index(drop=True, inplace=True)
    print(f'Option {name}: {df.shape}')

print(f'\nBest method distribution (Option A):')
print(df_a['best_method'].value_counts().to_string())

Option A: (86, 39)
Option B: (86, 17)
Option C: (86, 17)
Option AB: (86, 47)
Option AC: (86, 47)
Option Rand: (86, 17)

Best method distribution (Option A):
best_method
gmm          29
kmeans       26
autoenc      13
agg           7
dbscan        6
dictlearn     5


In [3]:
y_true = df_a['best_method'].values

baseline_kmeans = baseline_always('kmeans', y_true)
oracle_lse_val  = oracle_expected_lse(df_a)
always_kmeans_lse = df_a['LSE_kmeans'].mean()

de_souto = default_ranking_baseline(df_a)
print('=== Baselines ===')
print(f'  Always k-means          accuracy={baseline_kmeans:.3f}  expected_LSE={always_kmeans_lse:.3f}')
print(f'  de Souto default-ranking: best method = {de_souto["default_method"]}  '
      f'accuracy={de_souto["baseline_accuracy"]:.3f}  expected_LSE={de_souto["baseline_expected_lse"]:.3f}')
print(f'  Oracle                   accuracy=1.000  expected_LSE={oracle_lse_val:.3f}')
print()
print('  Average LSE per method (de Souto ranking source):')
for m, v in de_souto['avg_lse_vector'].items():
    print(f'    {m:10s}  {v:.4f}')

=== Baselines ===
  Always k-means          accuracy=0.302  expected_LSE=0.714
  de Souto default-ranking: best method = gmm  accuracy=0.337  expected_LSE=0.723
  Oracle                   accuracy=1.000  expected_LSE=0.798

  Average LSE per method (de Souto ranking source):
    kmeans      0.7135
    dbscan      0.5819
    agg         0.6914
    gmm         0.7227
    autoenc     0.7161
    dictlearn   0.6426


## kNN k-Sweep on Option A

In [4]:
X_a, y_a, feat_cols_a, ids_a = extract_Xy_clf(df_a)

k_candidates = [1, 3, 5, 7, 9, 11, 13]
k_results = {}
for k in k_candidates:
    pipe = build_classifier_candidates(best_k=k, random_state=SEED)['kNN']
    res  = loo_classify(pipe, X_a, y_a)
    k_results[k] = res['accuracy']
    print(f'  k={k:2d}  accuracy={res["accuracy"]:.3f}')

best_k = max(k_results, key=k_results.get)
print(f'\nBest k = {best_k}  (accuracy={k_results[best_k]:.3f})')

  k= 1  accuracy=0.267
  k= 3  accuracy=0.221
  k= 5  accuracy=0.279
  k= 7  accuracy=0.279
  k= 9  accuracy=0.372
  k=11  accuracy=0.314
  k=13  accuracy=0.326

Best k = 9  (accuracy=0.372)


## Classification LOO-CV — All Models × All Feature Options

For comparison to prior classification-based work.

In [5]:
clf_candidates = build_classifier_candidates(best_k=best_k, random_state=SEED)

options_clf = {
    'A (hand-crafted)' : df_a,
    'B (autoencoder)'  : df_b,
    'C (dictlearn)'    : df_c,
    'A+B'             : df_ab,
    'A+C (main claim)': df_ac,
    'Random (sanity)' : df_rand,
}

clf_results = {}
for opt_name, df_opt in options_clf.items():
    X_opt, y_opt, _, _ = extract_Xy_clf(df_opt)
    for model_name, pipe in clf_candidates.items():
        key = f'{opt_name} / {model_name}'
        res = loo_classify(pipe, X_opt, y_opt)
        clf_results[key] = res
        report_clf_results(key, res)

print('\nDone.')

  A (hand-crafted) / kNN                         acc=0.372  dist={'kmeans': 30, 'gmm': 41, 'agg': 5, 'autoenc': 10}
  A (hand-crafted) / LogReg                      acc=0.279  dist={'autoenc': 20, 'kmeans': 20, 'gmm': 18, 'agg': 12, 'dbscan': 6, 'dictlearn': 10}
  A (hand-crafted) / SVC-RBF                     acc=0.174  dist={'autoenc': 11, 'kmeans': 23, 'dbscan': 6, 'gmm': 16, 'agg': 18, 'dictlearn': 12}
  A (hand-crafted) / ExtraTrees                  acc=0.291  dist={'autoenc': 14, 'kmeans': 30, 'gmm': 23, 'agg': 5, 'dbscan': 6, 'dictlearn': 8}
  A (hand-crafted) / RF                          acc=0.419  dist={'autoenc': 11, 'kmeans': 30, 'gmm': 35, 'dbscan': 4, 'dictlearn': 2, 'agg': 4}
  A (hand-crafted) / MLP                         acc=0.291  dist={'autoenc': 13, 'kmeans': 27, 'dbscan': 4, 'gmm': 31, 'dictlearn': 5, 'agg': 6}
  B (autoencoder) / kNN                          acc=0.233  dist={'gmm': 38, 'kmeans': 25, 'autoenc': 22, 'agg': 1}
  B (autoencoder) / LogReg             

In [6]:
rows = []
for key, res in clf_results.items():
    opt, model = key.split(' / ')
    rows.append({'Option': opt, 'Model': model, 'Top-1 Acc': round(res['accuracy'], 3)})

rows.append({'Option': '-', 'Model': 'Always k-means',           'Top-1 Acc': round(baseline_kmeans, 3)})
rows.append({'Option': '-', 'Model': f'de Souto ({de_souto["default_method"]})', 'Top-1 Acc': round(de_souto['baseline_accuracy'], 3)})
rows.append({'Option': '-', 'Model': 'Oracle',                   'Top-1 Acc': 1.000})

summary_df = pd.DataFrame(rows)
print('=== Classification Results (sorted) ===')
print(summary_df.sort_values('Top-1 Acc', ascending=False).to_string(index=False))

=== Classification Results (sorted) ===
          Option          Model  Top-1 Acc
               -         Oracle      1.000
             A+B             RF      0.419
A (hand-crafted)             RF      0.419
A+C (main claim)             RF      0.419
A (hand-crafted)            kNN      0.372
               - de Souto (gmm)      0.337
A+C (main claim)     ExtraTrees      0.337
 Random (sanity)             RF      0.326
A+C (main claim)            kNN      0.326
             A+B            kNN      0.314
 B (autoencoder)            MLP      0.314
   C (dictlearn)            MLP      0.314
   C (dictlearn)            kNN      0.302
   C (dictlearn)        SVC-RBF      0.302
               - Always k-means      0.302
A (hand-crafted)     ExtraTrees      0.291
A+C (main claim)            MLP      0.291
A (hand-crafted)            MLP      0.291
A (hand-crafted)         LogReg      0.279
             A+B     ExtraTrees      0.279
 Random (sanity)            kNN      0.279
A+C (main clai

## Regression LOO-CV — Architecture A (Multi-Output)

Primary evaluation. Predicts the full LSE vector; recommended method = argmax.

In [7]:
reg_candidates = build_regressor_candidates(best_k=best_k, random_state=SEED)
X_a_reg, Y_a_reg, _, _ = extract_Xy_reg(df_a)

print('=== Architecture A: Multi-Output Regression (Option A features) ===')
print(f'{"Model":12s}  {"MAE":>6s}  {"SRC":>6s}  {"Top-1":>6s}  {"Top-2+tie":>9s}  {"E[LSE]":>7s}')
print('-' * 65)

reg_results_A = {}
for model_name, pipe in reg_candidates.items():
    res = loo_regress(pipe, X_a_reg, Y_a_reg, df_a)
    reg_results_A[model_name] = res
    print(f'{model_name:12s}  {res["mae_mean"]:6.4f}  {res["src"]:6.3f}  '
          f'{res["top1_accuracy"]:6.3f}  {res["top2_accuracy"]:9.3f}  {res["expected_lse"]:7.3f}')

print('-' * 65)
print(f'{"Always k-means":12s}  {"":>6s}  {"":>6s}  {baseline_kmeans:6.3f}  {"":>9s}  {always_kmeans_lse:7.3f}')
print(f'{"de Souto":12s}  {"":>6s}  {"":>6s}  {de_souto["baseline_accuracy"]:6.3f}  {"":>9s}  {de_souto["baseline_expected_lse"]:7.3f}')
print(f'{"Oracle":12s}  {"":>6s}  {"":>6s}  {1.000:6.3f}  {"":>9s}  {oracle_lse_val:7.3f}')

=== Architecture A: Multi-Output Regression (Option A features) ===
Model            MAE     SRC   Top-1  Top-2+tie   E[LSE]
-----------------------------------------------------------------
kNN           0.1581   0.425   0.326      0.523    0.710
RF            0.1345   0.408   0.326      0.500    0.734
ExtraTrees    0.1304   0.411   0.291      0.535    0.726
MLP           1.1467   0.304   0.291      0.430    0.708
-----------------------------------------------------------------
Always k-means                   0.302               0.714
de Souto                       0.337               0.723
Oracle                         1.000               0.798


## Regression LOO-CV — Architecture B (Per-Method, de Souto Style)

6 separate single-output regressors, one per method. Follows the architecture
that worked best in de Souto et al. 2008.

In [8]:
print('=== Architecture B: Per-Method Regression (Option A features) ===')
print(f'{"Model":12s}  {"MAE":>6s}  {"SRC":>6s}  {"Top-1":>6s}  {"Top-2+tie":>9s}  {"E[LSE]":>7s}')
print('-' * 65)

reg_results_B = {}
for model_name, pipe in reg_candidates.items():
    res = loo_regress_per_method(pipe, X_a_reg, Y_a_reg, df_a)
    reg_results_B[model_name] = res
    print(f'{model_name:12s}  {res["mae_mean"]:6.4f}  {res["src"]:6.3f}  '
          f'{res["top1_accuracy"]:6.3f}  {res["top2_accuracy"]:9.3f}  {res["expected_lse"]:7.3f}')

print('-' * 65)
best_A_name = min(reg_results_A, key=lambda n: reg_results_A[n]['mae_mean'])
best_B_name = min(reg_results_B, key=lambda n: reg_results_B[n]['mae_mean'])
print(f'\nBest Architecture A: {best_A_name}  MAE={reg_results_A[best_A_name]["mae_mean"]:.4f}')
print(f'Best Architecture B: {best_B_name}  MAE={reg_results_B[best_B_name]["mae_mean"]:.4f}')
winner = 'A' if reg_results_A[best_A_name]['mae_mean'] <= reg_results_B[best_B_name]['mae_mean'] else 'B'
print(f'Winner: Architecture {winner}')

=== Architecture B: Per-Method Regression (Option A features) ===
Model            MAE     SRC   Top-1  Top-2+tie   E[LSE]
-----------------------------------------------------------------
kNN           0.1581   0.425   0.326      0.523    0.710
RF            0.1333   0.435   0.291      0.581    0.727
ExtraTrees    0.1279   0.399   0.267      0.407    0.725
MLP           0.9189   0.340   0.337      0.395    0.701
-----------------------------------------------------------------

Best Architecture A: ExtraTrees  MAE=0.1304
Best Architecture B: ExtraTrees  MAE=0.1279
Winner: Architecture B


## Feature Representation Ablation

Compares all options for the best regression model (Architecture A, best model).
Key question: does A+C beat A alone? Does A+C beat random?

In [9]:
best_reg_model = min(reg_results_A, key=lambda n: reg_results_A[n]['mae_mean'])
best_reg_pipe  = reg_candidates[best_reg_model]

print(f'Using best model: {best_reg_model}')
print()
print(f'{"Option":20s}  {"MAE":>6s}  {"SRC":>6s}  {"Top-1":>6s}  {"Top-2+tie":>9s}  {"E[LSE]":>7s}')
print('-' * 72)

abl_results = {}
for opt_name, df_opt in options_clf.items():
    X_opt, Y_opt, _, _ = extract_Xy_reg(df_opt)
    res = loo_regress(best_reg_pipe, X_opt, Y_opt, df_opt)
    abl_results[opt_name] = res
    print(f'{opt_name:20s}  {res["mae_mean"]:6.4f}  {res["src"]:6.3f}  '
          f'{res["top1_accuracy"]:6.3f}  {res["top2_accuracy"]:9.3f}  {res["expected_lse"]:7.3f}')

print('-' * 72)
print(f'{"de Souto baseline":20s}  {"":>6s}  {"":>6s}  '
      f'{de_souto["baseline_accuracy"]:6.3f}  {"":>9s}  {de_souto["baseline_expected_lse"]:7.3f}')
print(f'{"Oracle":20s}  {"":>6s}  {"":>6s}  {1.000:6.3f}  {"":>9s}  {oracle_lse_val:7.3f}')

# Highlight the key comparisons
print('\n=== Key comparisons ===')
for pair in [('A (hand-crafted)', 'A+B'), ('A (hand-crafted)', 'A+C (main claim)'),
              ('A+C (main claim)', 'Random (sanity)')]:
    a_mae = abl_results[pair[0]]['mae_mean']
    b_mae = abl_results[pair[1]]['mae_mean']
    delta = a_mae - b_mae
    winner = pair[1] if delta > 0 else pair[0]
    print(f'  {pair[0]} vs {pair[1]}: ΔMAE={delta:+.4f} → {winner} wins')

Using best model: ExtraTrees

Option                   MAE     SRC   Top-1  Top-2+tie   E[LSE]
------------------------------------------------------------------------
A (hand-crafted)      0.1304   0.411   0.291      0.535    0.726
B (autoencoder)       0.1764   0.386   0.244      0.442    0.720
C (dictlearn)         0.1649   0.359   0.221      0.384    0.709
A+B                   0.1326   0.421   0.326      0.512    0.726
A+C (main claim)      0.1356   0.407   0.337      0.535    0.727
Random (sanity)       0.1909   0.346   0.267      0.372    0.684
------------------------------------------------------------------------
de Souto baseline                      0.337               0.723
Oracle                                 1.000               0.798

=== Key comparisons ===
  A (hand-crafted) vs A+B: ΔMAE=-0.0022 → A (hand-crafted) wins
  A (hand-crafted) vs A+C (main claim): ΔMAE=-0.0052 → A (hand-crafted) wins
  A+C (main claim) vs Random (sanity): ΔMAE=-0.0553 → A+C (main claim) wi

In [10]:
opt_names  = list(abl_results.keys())
maes       = [abl_results[n]['mae_mean'] for n in opt_names]
src_vals   = [abl_results[n]['src'] for n in opt_names]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

colors = ['#4C72B0' if 'A' in n and '(' not in n.split('A')[1][:2] else
          '#55A868' if '+C' in n else
          '#DD8452' if '+B' in n else
          '#999999' for n in opt_names]

ax1.bar(opt_names, maes, color=colors)
ax1.set_ylabel('MAE (lower = better)')
ax1.set_title('Regression MAE by Feature Option')
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=30, ha='right')

ax2.bar(opt_names, src_vals, color=colors)
ax2.set_ylabel('Spearman Rank Corr (higher = better)')
ax2.set_title('Ranking Quality (SRC) by Feature Option')
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=30, ha='right')

plt.tight_layout()
path = os.path.join(FIGS_DIR, 'ablation_regression.png')
fig.savefig(path, dpi=130)
plt.close()
print(f'Saved → {path}')

Saved → c:\MLResearch\outputs\figures\ablation_regression.png


## Save Best Models

In [11]:
# Best classification model (Option A)
best_clf_name = max(clf_candidates,
                    key=lambda n: clf_results[f'A (hand-crafted) / {n}']['accuracy'])
best_clf_pipe = clf_candidates[best_clf_name]
final_clf = clone(best_clf_pipe)
final_clf.fit(X_a, y_a)
clf_path = os.path.join(MODELS_DIR, 'meta_clf_optA.pkl')
with open(clf_path, 'wb') as f:
    pickle.dump({'pipeline': final_clf, 'feature_cols': feat_cols_a}, f)
print(f'Classifier saved → {clf_path}  ({best_clf_name})')

# Best regression model (Option A, Architecture A)
best_reg_name = min(reg_results_A, key=lambda n: reg_results_A[n]['mae_mean'])
best_reg_pipe = reg_candidates[best_reg_name]
final_reg = clone(best_reg_pipe)
X_a_r, Y_a_r, feat_cols_a_r, _ = extract_Xy_reg(df_a)
valid_mask = ~np.isnan(Y_a_r).any(axis=1)
final_reg.fit(X_a_r[valid_mask], Y_a_r[valid_mask])
reg_path = os.path.join(MODELS_DIR, 'meta_reg_optA.pkl')
with open(reg_path, 'wb') as f:
    pickle.dump({'pipeline': final_reg, 'feature_cols': feat_cols_a_r,
                 'lse_cols': LSE_COLS, 'method_names': METHOD_NAMES}, f)
print(f'Regressor  saved → {reg_path}  ({best_reg_name})')

print(f'\nPhase complete. Best classifier: {best_clf_name}, best regressor: {best_reg_name}')
print('Ready for 06_shap_analysis.ipynb')

Classifier saved → c:\MLResearch\outputs\models\meta_clf_optA.pkl  (RF)
Regressor  saved → c:\MLResearch\outputs\models\meta_reg_optA.pkl  (ExtraTrees)

Phase complete. Best classifier: RF, best regressor: ExtraTrees
Ready for 06_shap_analysis.ipynb
